# How to Calculate SPEI 

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from scipy.stats import gamma, norm, genextreme

### Step 1: Generate Synthetic Precipitation and Potential Evapotranspiration Data


In [ ]:
np.random.seed(42)
time = np.arange(0, 120)  # 10 years of monthly data
precip = np.random.gamma(shape=2, scale=2, size=len(time))  # Synthetic precipitation
pet = np.random.uniform(1, 3, size=len(time))  # Potential Evapotranspiration

In [ ]:
deficit = precip - pet  # Climatic water balance

In [ ]:
df = xr.Dataset(
    {"deficit": ("time", deficit)}, 
    coords={"time": time}
)

### Step 2: Fit a GEV Distribution to the Water Balance Data


In [ ]:
def fit_gev(data):
    data = data[~np.isnan(data)]  # Remove NaN values
    if len(data) == 0:  # Ensure valid GEV distribution values
        return np.nan, np.nan, np.nan  # Return NaN if no valid data
    try:
        shape, loc, scale = genextreme.fit(data)  # Fit GEV distribution
        if scale <= 0 or np.isnan(shape) or np.isnan(scale):  # Ensure valid parameters
            return np.nan, np.nan, np.nan
        return shape, loc, scale
    except Exception as e:
        print(f"GEV fitting error: {e}, data: {data}")
        return np.nan, np.nan, np.nan

### Step 3: Transform Deficit Data into CDF

In [ ]:
def compute_cdf(data, scale=3):
    scale = int(scale)  # Ensure scale is an integer
    cdf_values = np.full_like(data, np.nan)
    for i in range(scale, len(data)):
        subset = data[int(i - scale):int(i)]  # Explicitly cast indices to integers
        subset = subset[~np.isnan(subset)]  # Remove NaN values
        if len(subset) == 0:
            continue
        shape, loc, scale_fit = fit_gev(subset)  # Fit GEV distribution
        if np.isnan(shape) or np.isnan(scale_fit):
            continue
        cdf = genextreme.cdf(data[i], shape, loc=loc, scale=scale_fit)
        cdf_values[i] = np.clip(cdf, 1e-10, 1 - 1e-10)  # Clip CDF to avoid -inf/inf in norm.ppf()
    return cdf_values

df["CDF"] = ("time", compute_cdf(df.deficit.values, scale=3))

### Step 4: Convert CDF into Standard Normal Scores (SPEI)

In [ ]:
def cdf_to_spei(cdf_data):
    spei = np.full_like(cdf_data, np.nan)
    valid = ~np.isnan(cdf_data)
    spei[valid] = norm.ppf(cdf_data[valid])  # Convert CDF values to normal scores
    return spei

df["SPEI"] = ("time", cdf_to_spei(df.CDF.values))


### Step 5: Visualizations


In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(df.deficit.values[~np.isnan(df.deficit.values)], bins=20, alpha=0.7, label="Original Deficit Data")
plt.xlabel("Deficit Values")
plt.ylabel("Frequency")
plt.title("Histogram of Deficit Data")
plt.legend()

plt.subplot(1, 2, 2)
plt.hist(df.SPEI.values[~np.isnan(df.SPEI.values)], bins=20, alpha=0.7, label="Transformed SPEI Data")
plt.xlabel("SPEI Values")
plt.ylabel("Frequency")
plt.title("Histogram of SPEI")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df.time, df.deficit, label="Deficit", color='blue')
plt.xlabel("Time (months)")
plt.ylabel("Deficit")
plt.title("Deficit Time Series")
plt.legend()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(df.time, df.CDF, label="CDF", color='green')
plt.xlabel("Time (months")
plt.ylabel("CDF")
plt.title("CDF Time Series")
plt.legend()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(df.time, df.SPEI, label="SPEI (3-month scale)", color='red')
plt.axhline(0, color="k", linestyle="--")
plt.xlabel("Time (months")
plt.ylabel("SPEI")
plt.title("Standardized Precipitation-Evapotranspiration Index (SPEI)")
plt.legend()
plt.show()